Instalar librerias

In [1]:
%pip install pymongo python-dotenv


   ---------------------------------------- 0.0/826.2 kB ? eta -:--:--
   -------------------------------------- - 786.4/826.2 kB 7.1 MB/s eta 0:00:01
   ---------------------------------------- 826.2/826.2 kB 6.1 MB/s  0:00:00

   ---------------------------------------- 0/3 [python-dotenv]
   ---------------------------------------- 0/3 [python-dotenv]
   ------------- -------------------------- 1/3 [dnspython]
   ------------- -------------------------- 1/3 [dnspython]
   ------------- -------------------------- 1/3 [dnspython]
   ------------- -------------------------- 1/3 [dnspython]
   ------------- -------------------------- 1/3 [dnspython]
   ------------- -------------------------- 1/3 [dnspython]
   ------------- -------------------------- 1/3 [dnspython]
   ------------- -------------------------- 1/3 [dnspython]
   ------------- -------------------------- 1/3 [dnspython]
   ------------- -------------------------- 1/3 [dnspython]
   ------------- --------------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Conexión  segura a  MongoDB  / Lee la contraseña desde el .env, así que no queda escrita en el código.

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv
from pymongo import MongoClient

RAIZ = Path.cwd().parent                 # carpeta "Proyecto Final DB NoSQL"
load_dotenv(RAIZ / ".env")               # lee las contraseñas desde .env

uri = os.getenv("MONGO_URI_NLP")
if not uri:
    raise RuntimeError("No se encontró MONGO_URI_NLP. Revise que exista el archivo .env en la carpeta del proyecto.")

cliente = MongoClient(uri, serverSelectionTimeoutMS=5000)
db = cliente[os.getenv("MONGO_DB", "resenas_nlp")]
cliente.admin.command("ping")
print("Conectado a la base:", db.name)

Conectado a la base: resenas_nlp


Limpieza de datos

In [ ]:
from datetime import datetime

ESTRELLAS = {"ONE": 1, "TWO": 2, "THREE": 3, "FOUR": 4, "FIVE": 5}

def limpiar_texto(texto):
    """Quita la traducción automática que agrega Google y deja el texto original.""" 
    if not texto:
        return None
    if "(Original)" in texto:                      # reseña escrita en otro idioma
        texto = texto.split("(Original)", 1)[1]
    else:                                          # reseña en español + traducción al inglés
        texto = texto.split("(Translated by Google)", 1)[0]
    texto = texto.strip()
    return texto or None

def a_fecha(valor):
    """Convierte '2020-11-30T04:41:28.373236Z' en una fecha de Python (UTC)."""
    if not valor:
        return None
    return datetime.fromisoformat(valor.replace("Z", "+00:00"))

# Prueba rápida
print(limpiar_texto("El mejor express\n\n(Translated by Google)\nThe best express"))
print(a_fecha("2020-11-30T04:41:28.373236Z"))

El mejor express
2020-11-30 04:41:28.373236+00:00


Registro de Negocios

In [3]:
NEGOCIOS = [
    {"nombre": "Z Express CR",                    "google_location_id": "4452794751047616077",  "archivo": "z_express_cr.json"},
    {"nombre": "Marisquería La Casa del Camarón", "google_location_id": "6176225006541180530",  "archivo": "marisqueria_casa_del_camaron.json"},
    {"nombre": "ASM Construcción y Remodelación", "google_location_id": "14909255017997408569", "archivo": "asm_construccion.json"},
    {"nombre": "ASEMINSA",                        "google_location_id": "15277915664842449345", "archivo": "aseminsa.json"},
]

for n in NEGOCIOS:
    db.negocios.update_one(
        {"google_location_id": n["google_location_id"]},
        {"$setOnInsert": {
            "nombre": n["nombre"],
            "google_location_id": n["google_location_id"],
            "categoria_id": None,
            "activo": True,
            "creado_en": datetime.now().astimezone(),
        }},
        upsert=True,
    )
    n["_id"] = db.negocios.find_one({"google_location_id": n["google_location_id"]})["_id"]
    print(f"{n['nombre']:35} -> {n['_id']}")

Z Express CR                        -> 6b03d2d6d9baec43913c46bc
Marisquería La Casa del Camarón     -> 6b03d2d6d9baec43913c46bd
ASM Construcción y Remodelación     -> 6b03d2d6d9baec43913c46be
ASEMINSA                            -> 6b03d2d6d9baec43913c46bf


Cargar las reseñas 

In [4]:
import json
from bson import ObjectId
from pymongo.errors import BulkWriteError

CARPETA_DATOS = RAIZ / "resenas"

def cargar_negocio(negocio):
    ruta = CARPETA_DATOS / negocio["archivo"]
    with open(ruta, encoding="utf-8") as f:
        crudas = json.load(f)["reviews"]

    importacion_id = ObjectId()
    documentos, errores = [], []

    for r in crudas:
        try:
            respuesta = None
            if r.get("reviewReply"):
                respuesta = {
                    "comentario": limpiar_texto(r["reviewReply"].get("comment")) or "",
                    "fecha": a_fecha(r["reviewReply"].get("updateTime")),
                }
            documentos.append({
                "negocio_id": negocio["_id"],
                "importacion_id": importacion_id,
                "google_review_id": r["name"],
                "autor": r.get("reviewer", {}).get("displayName"),
                "estrellas": ESTRELLAS[r["starRating"]],
                "comentario": limpiar_texto(r.get("comment")),
                "fecha_creacion": a_fecha(r["createTime"]),
                "fecha_actualizacion": a_fecha(r.get("updateTime")),
                "respuesta": respuesta,
                "nlp": None,                       # se llenará en el análisis NLP
            })
        except Exception as e:
            errores.append(f"{r.get('name', '¿sin id?')}: {e}")

    cargadas, duplicadas = 0, 0
    if documentos:
        try:
            resultado = db.resenas.insert_many(documentos, ordered=False)
            cargadas = len(resultado.inserted_ids)
        except BulkWriteError as bwe:
            cargadas = bwe.details["nInserted"]
            for err in bwe.details["writeErrors"]:
                if err["code"] == 11000:           # ya existía (índice único)
                    duplicadas += 1
                else:
                    errores.append(err["errmsg"][:200])

    db.importaciones.insert_one({
        "_id": importacion_id,
        "negocio_id": negocio["_id"],
        "archivo": negocio["archivo"],
        "fecha": datetime.now().astimezone(),
        "estado": "completada" if not errores else "con_errores",
        "total_leidas": len(crudas),
        "total_cargadas": cargadas,
        "total_duplicadas": duplicadas,
        "errores": errores,
    })
    return len(crudas), cargadas, duplicadas, errores

for n in NEGOCIOS:
    leidas, cargadas, duplicadas, errores = cargar_negocio(n)
    print(f"{n['nombre']:35} leídas: {leidas:3}  cargadas: {cargadas:3}  duplicadas: {duplicadas:3}  errores: {len(errores)}")
    for e in errores:
        print("   ⚠", e)

Z Express CR                        leídas:  44  cargadas:  44  duplicadas:   0  errores: 0
Marisquería La Casa del Camarón     leídas: 151  cargadas: 151  duplicadas:   0  errores: 0
ASM Construcción y Remodelación     leídas:   9  cargadas:   9  duplicadas:   0  errores: 0
ASEMINSA                            leídas:   1  cargadas:   1  duplicadas:   0  errores: 0


Actualizar el resumen de cada negocio

In [5]:
for n in NEGOCIOS:
    resumen = list(db.resenas.aggregate([
        {"$match": {"negocio_id": n["_id"]}},
        {"$group": {"_id": None, "total": {"$sum": 1}, "promedio": {"$avg": "$estrellas"}}},
    ]))
    total = resumen[0]["total"] if resumen else 0
    promedio = round(resumen[0]["promedio"], 2) if resumen else None
    db.negocios.update_one({"_id": n["_id"]},
                           {"$set": {"total_resenas": total, "calificacion_promedio": promedio}})
    print(f"{n['nombre']:35} reseñas: {total:3}  promedio: {promedio}")

Z Express CR                        reseñas:  44  promedio: 4.61
Marisquería La Casa del Camarón     reseñas: 151  promedio: 4.7
ASM Construcción y Remodelación     reseñas:   9  promedio: 5.0
ASEMINSA                            reseñas:   1  promedio: 5.0


Verificación

In [6]:
print("Negocios:     ", db.negocios.count_documents({}))
print("Reseñas:      ", db.resenas.count_documents({}))
print("Con texto:    ", db.resenas.count_documents({"comentario": {"$ne": None}}))
print("Con respuesta:", db.resenas.count_documents({"respuesta": {"$ne": None}}))
print("Importaciones:", db.importaciones.count_documents({}))

print("\nEjemplo de reseña guardada:")
ejemplo = db.resenas.find_one({"comentario": {"$ne": None}, "respuesta": {"$ne": None}},
                              {"_id": 0, "autor": 1, "estrellas": 1, "fecha_creacion": 1,
                               "comentario": 1, "respuesta.comentario": 1})
for k, v in ejemplo.items():
    print(f"  {k}: {v}")

Negocios:      4
Reseñas:       205
Con texto:     108
Con respuesta: 57
Importaciones: 4

Ejemplo de reseña guardada:
  autor: Tatiana Salazar
  estrellas: 5
  comentario: El mejor express de Pérez Zeledón
  fecha_creacion: 2020-11-30 04:41:28.373000
  respuesta: {'comentario': 'Muchas gracias por tu calificación, Nos satisface saber que tu experiencia con nuestros servicios de mensajería han sido como los esperabas, siempre es un placer poder atenderte. Gracias.'}
